# read time era file -> grib or netcdf?

In [85]:
# Importations
import xarray as xr
import numpy as np
import numba as nb
import pandas as pd
import sys
import os
import time
import textwrap
import warnings
from scipy import interpolate
from clisops.core import subset
from pyproj import Transformer
from scipy.stats import spearmanr

In [21]:
# Load metadata of NetCDF files
ds = xr.open_dataset("D:/COMEPHORE/comephore_1km-1h_200105.nc", engine="netcdf4")

c:\Users\eving\AppData\Local\miniconda3\envs\evalprecextremes\Lib\site-packages\xarray\conventions.py:204: SerializationWarning: variable 'ERR' has multiple fill values {np.int32(-888), np.int32(65535)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
c:\Users\eving\AppData\Local\miniconda3\envs\evalprecextremes\Lib\site-packages\xarray\conventions.py:204: SerializationWarning: variable 'QUALIF' has multiple fill values {np.int32(-888), np.int32(255)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
c:\Users\eving\AppData\Local\miniconda3\envs\evalprecextremes\Lib\site-packages\xarray\conventions.py:204: SerializationWarning: variable 'RR' has multiple fill values {np.int32(-888), np.int32(65535)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)


In [86]:
lat_city=48.85341
lon_city=2.3488

# find regular lon/lat for the city
abs_dist = abs(ds.longitude-lon_city)+abs(ds.latitude-lat_city)
min_dist = np.where(abs_dist == abs_dist.min())
x_city = ds.X[min_dist[0]].values[0]
y_city = ds.Y[min_dist[1]].values[0]

# Create a boolean mask
# The parentheses are important because | (or) and & (and) have higher precedence than comparison operators in some contexts.
mask = (
    (ds["latitude"] >= (lat_city-0.1)) & (ds["latitude"] <= (lat_city+0.1)) &
    (ds["longitude"] >= (lon_city-0.1)) & (ds["longitude"] <= (lon_city+0.1))
)

# Apply the mask using .where()
# The drop=True argument removes data outside the selection bounds entirely
# instead of filling non-matching values with NaN.
ds_mask = ds.where(mask, drop=True)

In [94]:
ds_city = ds.sel(X=x_city,Y=y_city)
imax = int(ds_city.RR.argmax())
pr_3D = ds_mask.RR.isel(time=imax)

In [ ]:
# reshape pr data to 2D array (time, space) and compute spearman correlation matrix
pr_3D = ds_mask['RR'].values
pr_2D = np.reshape(pr_3D, (pr_3D.shape[0], pr_3D.shape[1]*pr_3D.shape[2]))

# remove nan values
notna = ~np.isnan(pr_2D).any(axis=0)
pr_2D = pr_2D[:,notna]

corr_matrix = spearmanr(pr_2D, axis=0,nan_policy ='omit').correlation

NameError: name 'stats' is not defined

In [67]:
prec_pixel = prec[:,int(size[1]/2),int(size[2]/2)]
max_index = int(prec_pixel.argmax())


In [72]:
# compute distance matrix
transformer = Transformer.from_crs("EPSG:4326", "EPSG:9794")
x,y = transformer.transform(selected_data.latitude,selected_data.longitude)

In [ ]:
s = size(x)
nx = s[0]
ny = s[1]
matrix_distance = np.zeros((nx,ny))
for i in range(nx):
    for j in range(ny):
        x1 = vecx[i]
        y1 = vecy[i]
        x2 = vecx[j]
        y2 = vecy[j]
        d = ((x2 - x1)**2 + (y2 - y1)**2)**0.5/1000  # distance in km
        matrix_distance[i,j] = d

AttributeError: 'function' object has no attribute 'proximity'